# Awaaz — Colab Data Preparation Runner
Mounts Google Drive, authenticates with Hugging Face, downloads candidate corpora (Common Voice, FLEURS, IndicVoices), converts to 16 kHz mono FLAC, balances manifests, and trains the SentencePiece tokenizer.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/awaaz/data

In [ ]:
# 2. Clone/Install Awaaz repository
!git clone https://github.com/Paramveersingh-S/Awaaz.git /content/awaaz_repo
%cd /content/awaaz_repo
!pip install -e .[pipeline,export]

In [ ]:
# 3. Read Hugging Face Token from Colab Secrets
import os

from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN successfully loaded.')
except Exception:
    print('HF_TOKEN secret not configured; public datasets will be used.')

In [ ]:
# 4. Prepare Datasets (dry-run with --max_items 50 or remove for full download)
!python scripts/prepare_hf.py --dataset_id google/fleurs --subset en_us --language en --split dev --output_dir /content/drive/MyDrive/awaaz/data/fleurs_en
!python scripts/prepare_hf.py --dataset_id google/fleurs --subset pa_in --language pa --split dev --output_dir /content/drive/MyDrive/awaaz/data/fleurs_pa

In [ ]:
# 5. Merge manifests and calculate language ratio
!python scripts/merge_manifests.py \
  --manifests /content/drive/MyDrive/awaaz/data/fleurs_en/dev_manifest.jsonl /content/drive/MyDrive/awaaz/data/fleurs_pa/dev_manifest.jsonl \
  --languages en pa \
  --repeats 1 1 \
  --output /content/drive/MyDrive/awaaz/data/combined_dev.jsonl